In [33]:
# !pip install -q \
# numpy==1.26.4 \
# pandas==2.2.2 \
# datasets==2.19.1 \
# transformers==4.55.4 \
# accelerate \
# evaluate \
# sentencepiece \
# scikit-learn \
# peft

In [34]:
import os
import time
import torch
import numpy as np
import pandas as pd

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

from peft import (
    LoraConfig,
    TaskType,
    get_peft_model
)

import evaluate

In [35]:
!git clone https://github.com/nyu-mll/GLUE-baselines.git

Cloning into 'GLUE-baselines'...
remote: Enumerating objects: 891, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 891 (delta 1), reused 3 (delta 1), pack-reused 886 (from 1)
Receiving objects: 100% (891/891), 1.48 MiB | 15.13 MiB/s, done.
Resolving deltas: 100% (610/610), done.


In [36]:
%cd GLUE-baselines

!python download_glue_data.py \
--data_dir glue_data \
--tasks CoLA

/content/GLUE-baselines/GLUE-baselines
	Completed!


In [37]:
!ls glue_data/CoLA

dev.tsv  original  test.tsv  train.tsv


In [38]:
train_df = pd.read_csv(
    "glue_data/CoLA/train.tsv",
    sep="\t",
    header=None
)

dev_df = pd.read_csv(
    "glue_data/CoLA/dev.tsv",
    sep="\t",
    header=None
)

train_df = train_df[[1,3]]
dev_df = dev_df[[1,3]]

train_df.columns = ["label","sentence"]
dev_df.columns = ["label","sentence"]

In [39]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(dev_df)

MODEL_NAME = "microsoft/deberta-v3-base"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

def tokenize_function(examples):

    return tokenizer(
        examples["sentence"],
        truncation=True,
        max_length=128
    )

train_dataset = train_dataset.map(
    tokenize_function,
    batched=True
)

val_dataset = val_dataset.map(
    tokenize_function,
    batched=True
)

train_dataset = train_dataset.rename_column(
    "label",
    "labels"
)

val_dataset = val_dataset.rename_column(
    "label",
    "labels"
)

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


Map:   0%|          | 0/8551 [00:00<?, ? examples/s]

Map:   0%|          | 0/1043 [00:00<?, ? examples/s]

In [40]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

for name, module in model.named_modules():

    if "query" in name.lower() or \
       "value" in name.lower():

        print(name)

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


deberta.encoder.layer.0.attention.self.query_proj
deberta.encoder.layer.0.attention.self.value_proj
deberta.encoder.layer.1.attention.self.query_proj
deberta.encoder.layer.1.attention.self.value_proj
deberta.encoder.layer.2.attention.self.query_proj
deberta.encoder.layer.2.attention.self.value_proj
deberta.encoder.layer.3.attention.self.query_proj
deberta.encoder.layer.3.attention.self.value_proj
deberta.encoder.layer.4.attention.self.query_proj
deberta.encoder.layer.4.attention.self.value_proj
deberta.encoder.layer.5.attention.self.query_proj
deberta.encoder.layer.5.attention.self.value_proj
deberta.encoder.layer.6.attention.self.query_proj
deberta.encoder.layer.6.attention.self.value_proj
deberta.encoder.layer.7.attention.self.query_proj
deberta.encoder.layer.7.attention.self.value_proj
deberta.encoder.layer.8.attention.self.query_proj
deberta.encoder.layer.8.attention.self.value_proj
deberta.encoder.layer.9.attention.self.query_proj
deberta.encoder.layer.9.attention.self.value_proj


LoRA:

In [41]:
sora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=[
        "query_proj",
        "value_proj"
    ]
)

In [42]:
# !pip uninstall -y torchao
# !pip install -U torchao

In [43]:
model = get_peft_model(
    model,
    sora_config
)

In [44]:
model.print_trainable_parameters()

trainable params: 296,450 || all params: 184,720,132 || trainable%: 0.1605


In [45]:
for name, param in model.named_parameters():
    if "lora" in name.lower():
        print(name)

base_model.model.deberta.encoder.layer.0.attention.self.query_proj.lora_A.default.weight
base_model.model.deberta.encoder.layer.0.attention.self.query_proj.lora_B.default.weight
base_model.model.deberta.encoder.layer.0.attention.self.value_proj.lora_A.default.weight
base_model.model.deberta.encoder.layer.0.attention.self.value_proj.lora_B.default.weight
base_model.model.deberta.encoder.layer.1.attention.self.query_proj.lora_A.default.weight
base_model.model.deberta.encoder.layer.1.attention.self.query_proj.lora_B.default.weight
base_model.model.deberta.encoder.layer.1.attention.self.value_proj.lora_A.default.weight
base_model.model.deberta.encoder.layer.1.attention.self.value_proj.lora_B.default.weight
base_model.model.deberta.encoder.layer.2.attention.self.query_proj.lora_A.default.weight
base_model.model.deberta.encoder.layer.2.attention.self.query_proj.lora_B.default.weight
base_model.model.deberta.encoder.layer.2.attention.self.value_proj.lora_A.default.weight
base_model.model.debe

In [46]:
sora_masks = {}

In [47]:
from transformers import TrainerCallback
import torch

class SoRAPruningCallback(TrainerCallback):

    def __init__(self, final_sparsity=0.8):
        self.final_sparsity = final_sparsity

    def on_epoch_end(self, args, state, control, model=None, **kwargs):

        current_epoch = int(state.epoch)

        total_epochs = args.num_train_epochs

        sparsity = (
            self.final_sparsity *
            current_epoch /
            total_epochs
        )

        print(f"\nTarget sparsity={sparsity:.2f}")

        for name, param in model.named_parameters():

            if (
                "lora_A" in name or
                "lora_B" in name
            ):

                with torch.no_grad():

                    flat = param.abs().flatten()

                    k = int(
                        sparsity *
                        flat.numel()
                    )

                    if k <= 0:
                        continue

                    threshold = torch.kthvalue(
                        flat,
                        k
                    ).values

                    mask = (
                        param.abs()
                        > threshold
                    )

                    sora_masks[name] = mask

                    param.mul_(mask)

In [48]:
class MaskEnforcementCallback(TrainerCallback):

    def on_step_end(
        self,
        args,
        state,
        control,
        model=None,
        **kwargs
    ):

        for name, param in model.named_parameters():

            if name in sora_masks:

                with torch.no_grad():

                    param.mul_(
                        sora_masks[name]
                    )

In [49]:
metric = evaluate.load(
    "glue",
    "cola"
)

In [50]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    preds = np.argmax(
        logits,
        axis=-1
    )

    return metric.compute(
        predictions=preds,
        references=labels
    )

In [51]:
training_args = TrainingArguments(
    output_dir="./lora_cola",

    learning_rate=2e-4,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    num_train_epochs=10,

    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="matthews_correlation",

    greater_is_better=True,

    logging_steps=50,

    report_to="none"
)

In [52]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[
    SoRAPruningCallback(
        final_sparsity=0.8
    ),
    MaskEnforcementCallback()
    ]
)

/tmp/ipykernel_2007/2371016613.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [53]:
start = time.time()

trainer.train()

end = time.time()

training_time = (end - start) / 60

print(
    f"Training Time: {training_time:.2f} minutes"
)

Epoch,Training Loss,Validation Loss,Matthews Correlation
1,0.371500,0.353239,0.631427
2,0.337400,0.394537,0.630691
3,0.343200,0.376698,0.660235
4,0.283900,0.431320,0.644507
5,0.254600,0.405412,0.662712
6,0.277800,0.394686,0.662769
7,0.253600,0.397019,0.670035
8,0.216200,0.384017,0.657795
9,0.218200,0.397581,0.642340
10,0.249500,0.370138,0.614371



Target sparsity=0.08

Target sparsity=0.16

Target sparsity=0.24

Target sparsity=0.32

Target sparsity=0.40

Target sparsity=0.48

Target sparsity=0.56

Target sparsity=0.64

Target sparsity=0.72

Target sparsity=0.80
Training Time: 8.46 minutes


In [62]:
for name, param in model.named_parameters():

    if name in sora_masks:

        surviving = (
            param != 0
        ).sum().item()

        total = param.numel()

        print(
            name,
            surviving,
            "/",
            total
        )

base_model.model.deberta.encoder.layer.0.attention.self.query_proj.lora_A.default.weight 2002 / 6144
base_model.model.deberta.encoder.layer.0.attention.self.query_proj.lora_B.default.weight 421 / 6144
base_model.model.deberta.encoder.layer.0.attention.self.value_proj.lora_A.default.weight 2060 / 6144
base_model.model.deberta.encoder.layer.0.attention.self.value_proj.lora_B.default.weight 84 / 6144
base_model.model.deberta.encoder.layer.1.attention.self.query_proj.lora_A.default.weight 2098 / 6144
base_model.model.deberta.encoder.layer.1.attention.self.query_proj.lora_B.default.weight 399 / 6144
base_model.model.deberta.encoder.layer.1.attention.self.value_proj.lora_A.default.weight 2073 / 6144
base_model.model.deberta.encoder.layer.1.attention.self.value_proj.lora_B.default.weight 78 / 6144
base_model.model.deberta.encoder.layer.2.attention.self.query_proj.lora_A.default.weight 2101 / 6144
base_model.model.deberta.encoder.layer.2.attention.self.query_proj.lora_B.default.weight 377 / 61

In [73]:
import torch

FINAL_SPARSITY = 0.50

all_weights = []

for name, param in model.named_parameters():

    if (
        "lora_A" in name or
        "lora_B" in name
    ):

        all_weights.append(
            param.abs().flatten()
        )

all_weights = torch.cat(all_weights)

k = int(
    FINAL_SPARSITY *
    all_weights.numel()
)

global_threshold = torch.kthvalue(
    all_weights,
    k
).values

for name, param in model.named_parameters():

    if (
        "lora_A" in name or
        "lora_B" in name
    ):

        with torch.no_grad():

            mask = (
                param.abs()
                > global_threshold
            )

            param.mul_(mask)

print("Global final pruning complete")

Global final pruning complete


In [74]:
total_lora = 0
nonzero_lora = 0

for name, param in model.named_parameters():

    if (
        "lora_A" in name or
        "lora_B" in name
    ):

        total_lora += param.numel()

        nonzero_lora += (
            param != 0
        ).sum().item()

print(
    "Effective Sparsity:",
    1 - nonzero_lora / total_lora
)

print(
    "Remaining Parameters:",
    nonzero_lora
)

Effective Sparsity: 0.7999979654947916
Remaining Parameters: 58983


In [75]:
compression_ratio = (
    total_lora /
    nonzero_lora
)

print(
    "Compression Ratio:",
    compression_ratio
)

Compression Ratio: 4.999949137887188


In [76]:
results = trainer.evaluate()

print(results)

{'eval_loss': 0.5238922834396362, 'eval_matthews_correlation': 0.3795768176897039, 'eval_runtime': 3.6056, 'eval_samples_per_second': 289.268, 'eval_steps_per_second': 18.305, 'epoch': 10.0}


In [67]:
trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

lora_results = {
    "Method": "LoRA",
    "MCC": results["eval_matthews_correlation"],
    "Trainable_Params": trainable_params,
    "Rank": 8,
    "Training_Time_Min": training_time
}

print(lora_results)

{'Method': 'LoRA', 'MCC': 0.3795768176897039, 'Trainable_Params': 296450, 'Rank': 8, 'Training_Time_Min': 8.46392127275467}


In [68]:
import torch
import numpy as np

effective_ranks = []

for name, module in model.named_modules():

    if hasattr(module, "lora_A") and hasattr(module, "lora_B"):

        A = module.lora_A["default"].weight.data
        B = module.lora_B["default"].weight.data

        delta_w = B @ A

        s = torch.linalg.svdvals(delta_w)

        threshold = 0.01 * s.max()

        eff_rank = (s > threshold).sum().item()

        effective_ranks.append(eff_rank)

        print(f"{name}: effective rank = {eff_rank}")

print("\nAverage Effective Rank:", np.mean(effective_ranks))

base_model.model.deberta.encoder.layer.0.attention.self.query_proj: effective rank = 8
base_model.model.deberta.encoder.layer.0.attention.self.value_proj: effective rank = 8
base_model.model.deberta.encoder.layer.1.attention.self.query_proj: effective rank = 8
base_model.model.deberta.encoder.layer.1.attention.self.value_proj: effective rank = 8
base_model.model.deberta.encoder.layer.2.attention.self.query_proj: effective rank = 8
base_model.model.deberta.encoder.layer.2.attention.self.value_proj: effective rank = 8
base_model.model.deberta.encoder.layer.3.attention.self.query_proj: effective rank = 8
base_model.model.deberta.encoder.layer.3.attention.self.value_proj: effective rank = 8
base_model.model.deberta.encoder.layer.4.attention.self.query_proj: effective rank = 8
base_model.model.deberta.encoder.layer.4.attention.self.value_proj: effective rank = 8
base_model.model.deberta.encoder.layer.5.attention.self.query_proj: effective rank = 8
base_model.model.deberta.encoder.layer.5.at